# Total Q4 2024 by channel

*Data Analysis in Python · Topic 4.3*

The CMO wants one table for the fourth quarter of 2024: each channel's sessions, transactions, revenue and conversion rate, best channel first, and the paid search campaigns that brought in the most revenue.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that builds `sessions` (lesson 3.'s steps) and the cell that adds its `channel_group` column (topic 4.2.'s steps).
2. In the first cell marked ✏️, create `by_channel`: one row per channel group, with the channel groups as the index, and the columns:
   - `sessions`, `transactions` and `revenue`: each group's totals
   - `conversion_rate`: transactions divided by sessions, for the group

   Sort it by revenue, highest first.
3. In the second cell marked ✏️, create:
   - `top_channel`: the name of the channel group with the most revenue
   - `email_rate`: Email's conversion rate, from `by_channel`
4. In the last cell marked ✏️, create `top_search_campaigns`: the three Paid Search campaigns with the most revenue, as a Series with the campaign names as the index and their revenue as the values.
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `by_channel` has nine rows, one per channel group, with `sessions`, `transactions`, `revenue` and `conversion_rate`, sorted by revenue: Direct first, with $207,788.92. Its sessions add up to 354,731. `top_channel` is `"Direct"` and `email_rate` is about `0.0689`. `top_search_campaigns` has three campaigns, led by `2024_us_search_shopping` with $24,952.78.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = ["2024-10", "2024-11", "2024-12"]
START, END = "2024-10-01", "2024-12-31"

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
# Channel groups, with topic 4.2's steps and Foundations topic 12.3's function. Run it, and leave it as it is.
SOCIAL_SITES = {"facebook.com", "m.facebook.com", "l.facebook.com", "lm.facebook.com", "instagram.com", "l.instagram.com"}


def clean(value):
    """Return the value lowercased and stripped, or "" for None."""
    if value is None:
        return ""
    return value.strip().lower()


def channel_group(source, medium):
    """Return the channel group of a session's source and medium, with Foundations topic 12.3's rules."""
    source = clean(source)
    medium = clean(medium)
    if source == "(direct)" and medium == "(none)":
        return "Direct"
    if medium == "email":
        return "Email"
    if medium == "cpc":
        return "Paid Search"
    if medium == "display":
        return "Display"
    if medium == "paid_social":
        return "Paid Social"
    if medium == "video":
        return "Paid Video"
    if medium == "organic":
        return "Organic Search"
    if medium == "referral" and source in SOCIAL_SITES:
        return "Organic Social"
    if medium == "referral":
        return "Referral"
    return "Unassigned"


def row_channel_group(row):
    """Return the channel group of one row of the sessions table."""
    return channel_group(row["session_source"], row["session_medium"])


sessions["channel_group"] = sessions.apply(row_channel_group, axis=1)
sessions["channel_group"].value_counts()

## The channel report

In [ ]:
# ✏️ Create by_channel, sorted by revenue


In [ ]:
# ✏️ Create top_channel and email_rate


## Paid search campaigns

In [ ]:
# ✏️ Create top_search_campaigns


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    by_channel = found.get("by_channel")
    columns = ["sessions", "transactions", "revenue", "conversion_rate"]
    if not isinstance(by_channel, pd.DataFrame):
        results.append((False, "by_channel isn't a DataFrame yet. Group sessions by channel_group, and total sessions, transactions and revenue with agg()."))
    elif len(by_channel) != 9:
        results.append((False, f"by_channel has {len(by_channel):,} rows, but there are nine channel groups. Group by channel_group only."))
    elif missing := [column for column in columns if column not in by_channel.columns]:
        results.append((False, f"by_channel has no {', '.join(missing)} column. Add conversion_rate after aggregating: by_channel[\"transactions\"] / by_channel[\"sessions\"]."))
    elif "Email" not in by_channel.index:
        results.append((False, "by_channel's index should be the channel groups, so by_channel.loc[\"Email\"] works. Group without as_index=False here."))
    elif abs(by_channel.loc["Direct", "sessions"] - 298.47) < 0.01:
        results.append((False, "by_channel's Direct sessions is 298.47: the mean sessions per row. Total each group with \"sum\"."))
    elif by_channel["sessions"].sum() != 354731 or abs(by_channel["revenue"].sum() - 790894.92) > 0.005:
        results.append((False, f"by_channel's sessions add up to {by_channel['sessions'].sum():,}, but sessions has 354,731. Every row should be in a group, and every group summed."))
    elif abs(by_channel.loc["Email", "conversion_rate"] - 0.0666) < 0.00005:
        results.append((False, "Email's conversion_rate is 0.0666: the mean of the rows' rates. Divide the group's total transactions by its total sessions, after aggregating."))
    elif not ((by_channel["conversion_rate"] - by_channel["transactions"] / by_channel["sessions"]).abs() < 1e-9).all():
        results.append((False, "by_channel's conversion_rate should be each group's transactions over its sessions, as a fraction."))
    elif by_channel.index[0] != "Direct" or not by_channel["revenue"].is_monotonic_decreasing:
        results.append((False, f"by_channel should be sorted by revenue, highest first, so Direct comes first. Its first row is {by_channel.index[0]}."))
    else:
        results.append((True, "by_channel has the nine groups, best revenue first, and its sessions add up to the quarter's 354,731."))

    top_channel = found.get("top_channel")
    email_rate = found.get("email_rate")
    if top_channel == "Organic Search":
        results.append((False, "top_channel is Organic Search, the group with the most sessions. The question is revenue: Direct."))
    elif top_channel != "Direct":
        results.append((False, f"top_channel should be the name of the group with the most revenue, by_channel.index[0] once it's sorted. It is {top_channel!r}."))
    elif isinstance(email_rate, (pd.DataFrame, pd.Series)):
        results.append((False, f"email_rate is a {type(email_rate).__name__}, not a number. Take one value: by_channel.loc[\"Email\", \"conversion_rate\"]."))
    elif pd.api.types.is_number(email_rate) and abs(email_rate - 0.068890) < 0.00005:
        results.append((True, f"Direct brought in the most revenue in Q4 2024, and Email converted {email_rate:.2%} of its sessions."))
    else:
        results.append((False, f"email_rate should be by_channel.loc[\"Email\", \"conversion_rate\"], about 0.0689. It is {email_rate!r}."))

    top = found.get("top_search_campaigns")
    expected = {"2024_us_search_shopping": 24952.78, "2024_us_search_generic": 19910.78, "2024_us_search_brand": 18803.06}
    if isinstance(top, pd.DataFrame):
        results.append((False, "top_search_campaigns is a DataFrame. Pick the revenue column before nlargest(), so it's a Series: ...groupby(\"session_campaign\")[\"revenue\"].sum().nlargest(3)."))
    elif not isinstance(top, pd.Series):
        results.append((False, "top_search_campaigns isn't a Series yet. Keep the Paid Search rows, group them by session_campaign, total their revenue and keep the three largest."))
    elif len(top) != 3:
        results.append((False, f"top_search_campaigns has {len(top):,} values, but it should have three."))
    elif set(top.index) != set(expected):
        results.append((False, f"top_search_campaigns has {list(top.index)}, but the three Paid Search campaigns with the most revenue are {list(expected)}. Keep only Paid Search rows, and total revenue per campaign."))
    elif any(abs(top[name] - value) > 0.005 for name, value in expected.items()):
        results.append((False, "top_search_campaigns has the right campaigns, but not their total revenue. Sum revenue per campaign."))
    else:
        results.append((True, "Paid search's top three: 2024_us_search_shopping ($24,952.78), generic and brand."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
by_channel = sessions.groupby("channel_group").agg({"sessions": "sum", "transactions": "sum", "revenue": "sum"})
by_channel["conversion_rate"] = by_channel["transactions"] / by_channel["sessions"]
by_channel = by_channel.sort_values("revenue", ascending=False)
by_channel["sessions"].sum() == sessions["sessions"].sum(), by_channel
```

```python
top_channel = by_channel.index[0]
email_rate = by_channel.loc["Email", "conversion_rate"]
top_channel, email_rate
```

```python
paid_search = sessions[sessions["channel_group"] == "Paid Search"]
top_search_campaigns = paid_search.groupby("session_campaign")["revenue"].sum().nlargest(3)
top_search_campaigns
```

</details>